# Self-Supervised 2D+3D Hybrid GraphGPS for Multi-Objective ADMET Profiling & Lead Optimization
### Engineering Semester ML Course Project — Interactive Demonstration Notebook

This notebook demonstrates the complete end-to-end computational drug discovery pipeline:
1. **Curated Real-World TDC + ZINC250k Dataset & 2D+3D Graph Featurization**
2. **Hybrid Local+Global GraphGPS Architecture & Fine-Tuned Model Loading**
3. **Held-Out Unseen-Scaffold Benchmark & Ablation Comparison**
4. **Multi-Generation Evolutionary Lead Optimization (SMIRKS + BRICS)**
5. **Multi-Objective Pareto Frontier Visualization**
6. **Atom-Level Explainability (XAI) Gradient Attribution Maps**

In [ ]:
import io
import json
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import Draw
from torch_geometric.data import Batch

from src.data.dataset import smiles_to_graph_data
from src.models.graph_transformer import MolecularGraphTransformer
from src.models.multi_objective_heads import MultiObjectiveADMETModel
from src.optimization.lead_mutator import MolecularLeadMutator
from src.optimization.pareto_scorer import MultiObjectiveScorer, ParetoFrontier
from src.utils.chemistry import (
    compute_exact_qed_sa,
    compute_lipinski_and_pains,
    compute_tanimoto_similarity,
    suppress_rdkit_warnings,
)
from src.utils.explainability import _draw_mol_with_saliency, compute_atom_saliency
from src.utils.seed import get_device, set_seed

set_seed(42)
suppress_rdkit_warnings()
device = get_device()
print(f'Using compute device: {device}')

## 1. Curated Real-World Dataset & 2D+3D Graph Featurization

In [ ]:
with open('../data/processed/dataset_stats.json', 'r', encoding='utf-8') as f:
    stats = json.load(f)

print(f"Unlabeled SSL Molecules (ZINC250k): {stats['unlabeled_ssl_count']}")
print(f"Multi-Task ADMET Compounds (TDC):   {stats['admet_total_count']} ({stats['unique_scaffolds']} unique Murcko scaffolds)")
for sp in ['train', 'val', 'test']:
    s = stats['splits'][sp]
    print(f"  {sp.upper():<5} -> {s['count']} molecules ({s['unique_scaffolds']} unique scaffolds)")

seed_smiles = 'CC(=O)Oc1ccccc1C(=O)O'  # Aspirin
g = smiles_to_graph_data(seed_smiles, compute_pe=True, pe_dim=8, use_3d=True)
print('\nAspirin 2D+3D Graph Representation:')
print(f'  Heavy Atom Nodes: {g.num_nodes}')
print(f'  Node Features x:  {tuple(g.x.shape)} (Chemical + 3D centroid distance)')
print(f'  Edge Attr:        {tuple(g.edge_attr.shape)} (Bond chemistry + 3D bond lengths)')
print(f'  LapPE + RWSE PE:  {tuple(g.pe.shape)} (Sign-canonicalized Laplacian + Random Walk)')
print(f'  Global 3D Geom:   {tuple(g.geom_3d.shape)} (NPR1, NPR2, Asphericity, RoG)')

## 2. Load Fine-Tuned Hybrid Local+Global GraphGPS Model

In [ ]:
backbone = MolecularGraphTransformer(hidden_dim=128, num_layers=3, heads=4, dropout=0.1).to(device)
model = MultiObjectiveADMETModel(backbone, hidden_dim=128, dropout=0.1).to(device)

ckpt_path = '../checkpoints/admet_finetuned_model.pth'
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
model.eval()
print(f'Loaded verified SSL-GraphGPS ADMET checkpoint: {ckpt_path}')

## 3. Held-Out Unseen-Scaffold Benchmark & Ablation Table

In [ ]:
with open('../results/metrics_summary.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)

rows = []
for m_name, m in metrics.items():
    rows.append({
        'Model': m_name,
        'Bioavail ROC-AUC': round(m['bioavailability']['roc_auc'], 4),
        'BBB ROC-AUC': round(m['bbb']['roc_auc'], 4),
        'hERG ROC-AUC': round(m['herg']['roc_auc'], 4),
        'Macro ROC-AUC': round(m['aggregate']['macro_roc_auc'], 4),
        'QED R2': round(m['qed']['r2'], 4),
        'SAScore R2': round(m['sascore']['r2'], 4),
    })
pd.DataFrame(rows)

## 4. Multi-Generation Evolutionary Lead Optimization (Aspirin Case Study)

In [ ]:
mutator = MolecularLeadMutator()
scorer = MultiObjectiveScorer(target_mode='peripheral')

@torch.no_grad()
def batch_score(smiles_batch):
    graphs, valid_sm = [], []
    for smi in smiles_batch:
        gd = smiles_to_graph_data(smi, compute_pe=True, pe_dim=8, use_3d=True)
        if gd is not None:
            graphs.append(gd)
            valid_sm.append(gd.smiles)
    b = Batch.from_data_list(graphs).to(device)
    preds = model(b.x, b.edge_index, b.edge_attr, b.batch, b.pe, b.geom_3d)
    out = []
    for i, smi in enumerate(valid_sm):
        q_rd, sa_rd = compute_exact_qed_sa(smi)
        mc = compute_lipinski_and_pains(smi)
        prof = {
            'smiles': smi,
            'bioavailability': float(preds['bioavailability'][i].item()),
            'bbb': float(preds['bbb'][i].item()),
            'herg': float(preds['herg'][i].item()),
            'qed': float(preds['qed'][i].item()),
            'sascore': float(preds['sascore'][i].item()),
            'qed_rdkit': round(q_rd, 4),
            'sascore_rdkit': round(sa_rd, 4),
            'tanimoto_sim': round(compute_tanimoto_similarity(seed_smiles, smi), 3),
            'pains_pass': mc['pains_pass'],
        }
        prof['desirability'] = scorer.compute_composite_desirability(prof)
        out.append(prof)
    return out

results = mutator.evolve_leads(seed_smiles, score_fn=batch_score, generations=2, pop_size=35, top_parents=6)
ParetoFrontier.assign_pareto_ranks(results, target_mode='peripheral')
df_res = pd.DataFrame(results).sort_values(by=['pareto_rank', 'desirability'], ascending=[True, False])
df_res[['smiles', 'pareto_rank', 'bioavailability', 'herg', 'bbb', 'qed', 'sascore', 'tanimoto_sim', 'desirability']].head(10)

## 5. Multi-Objective Pareto Frontier & Top Analogue Structures

In [ ]:
pareto_f1 = [r for r in results if r['pareto_rank'] == 1]
pareto_smiles = {p['smiles'] for p in pareto_f1}
seed_canon = Chem.MolToSmiles(Chem.MolFromSmiles(seed_smiles), canonical=True, isomericSmiles=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for r in results:
    if r['smiles'] == seed_canon:
        axes[0].scatter(r['herg'], r['bioavailability'], c='#DC2626', marker='*', s=240, edgecolors='k', zorder=5, label='Seed (Aspirin)')
    elif r['smiles'] in pareto_smiles:
        axes[0].scatter(r['herg'], r['bioavailability'], c='#1D4ED8', s=100, edgecolors='k', zorder=4)
    else:
        axes[0].scatter(r['herg'], r['bioavailability'], c='#94A3B8', s=45, alpha=0.65)

axes[0].set_title('Bioavailability vs. hERG Hazard Pareto Frontier')
axes[0].set_xlabel('Predicted hERG Cardiotoxicity Hazard (Lower is Safer)')
axes[0].set_ylabel('Predicted Bioavailability Score % (Higher is Better)')
axes[0].grid(True, linestyle='--', alpha=0.4)

top4 = df_res.head(4)['smiles'].tolist()
mols = [Chem.MolFromSmiles(s) for s in top4]
legends = [f"Rank #{i+1} | Desir={row['desirability']:.3f}" for i, (_, row) in enumerate(df_res.head(4).iterrows())]
pil_img = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(300, 220), legends=legends)
buf = io.BytesIO()
pil_img.save(buf, format='PNG')
buf.seek(0)
axes[1].imshow(plt.imread(buf))
axes[1].axis('off')
axes[1].set_title('Top Pareto-Optimal Aspirin Analogues')
plt.tight_layout()
plt.show()

## 6. Atom-Level Model Explainability (XAI Gradient Saliency)

In [ ]:
xai_targets = [
    ('Aspirin (hERG Attribution)', seed_smiles, 'herg'),
    ('Caffeine (BBB Attribution)', 'Cn1c(=O)c2c(ncn2C)n(C)c1=O', 'bbb'),
]
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.5))
for ax, (title, smi, task) in zip(axes, xai_targets):
    mol, scores, pred = compute_atom_saliency(model, smi, target_task=task, device=device)
    ax.imshow(_draw_mol_with_saliency(mol, scores))
    ax.axis('off')
    ax.set_title(f"{title}\nPred {task.upper()}: {pred[task]:.3f}")
plt.tight_layout()
plt.show()